# BSTMap Lab: Office Hours Booking

This is a short, interactive lab to test-drive the `BSTMap` you're
building this week in `bst_map.py`. Like the HashMap lab, it's completely
separate from your repository, just a safe place to click buttons and
watch your binary search tree work.

No installation needed, everything runs in your browser.

## Workflow

```
[1] Setup -> [2] Part 1 -> [3] Part 2 -> [4] Part 3 -> [5] Test with the app -> [6] Tasks + screenshots -> [7] Submit
```

| Step | What you do | You're done when... |
|---|---|---|
| 1. Setup | Run the setup cell once | No errors, nothing else to do |
| 2. Part 1 | Fill in `BSTMap` (the tree itself) | Every method has real code, no `NotImplementedError` left |
| 3. Part 2 | Fill in `OrderedBSTMap` (floor, ceiling, range) | Same: no `NotImplementedError` left |
| 4. Part 3 | Fill in `can_book` and `next_free_slot` (the booking rules) | Same: no `NotImplementedError` left |
| 5. Test | Scroll to the booking app and click around | Buttons do what they should instead of saying "not implemented" |
| 6. Tasks | Work through the 10 Lab Tasks near the bottom using the buttons | You have a screenshot + written answer for each task |
| 7. Submit | Attach your screenshots and answers | N/A |

**Important:** the booking app at the bottom always runs, even before
you've implemented anything. Click any button right now if you want, it'll
just tell you which method isn't implemented yet. As you fill in each
method and re-run that cell, the matching button starts working, with no
other changes needed: the app notices you re-ran a cell and rebuilds the
schedule with your new code. One catch: `OrderedBSTMap` is built on top of
`BSTMap`, so whenever you change Part 1, re-run the Part 2 cell too.

There's no separate check cell: the app itself is how you verify your
work. It redraws your tree after every click, checks that it's still a
valid BST, and double-checks every answer your methods return against the
tree. If something disagrees, the action log says so.

In [1]:
# Setup: just run this once, nothing to edit here.
try:
    import ipywidgets as widgets
except ImportError:
    %pip install -q ipywidgets
    import ipywidgets as widgets

from IPython.display import display

# Office hours run from 13:00 to 17:00. Times are stored as minutes since
# midnight, so 13:00 is 780 and 17:00 is 1020.
OPEN = 13 * 60
CLOSE = 17 * 60


def to_minutes(hhmm: str) -> int:
    """'13:45' -> 825 (minutes since midnight)."""
    hours, minutes = hhmm.split(":")
    return int(hours) * 60 + int(minutes)


def fmt(minutes: int) -> str:
    """825 -> '13:45'."""
    return f"{minutes // 60:02d}:{minutes % 60:02d}"

## The problem: Office Hours Booking

Your instructor holds office hours every Tuesday from **13:00 to 17:00**.
Students book a meeting by picking a start time and a length (10, 15, 20
or 30 minutes), and the booking system has to keep all of those meetings
straight. It needs to:

| The booking system wants to...                 | What it uses                                  | Part |
|--------------------------------------------------|-----------------------------------------------|------|
| Book, look up, or cancel a meeting               | `__setitem__`, `__getitem__`, `__delitem__`   | 1    |
| Print the afternoon's schedule in time order     | `__iter__`                                    | 1    |
| Say who is in the room at 15:05, and who's next  | `floor_key`, `ceiling_key`                    | 2    |
| List everyone coming between 15:00 and 15:30     | `range_keys`                                  | 2    |
| Refuse a meeting that clashes with another one   | `can_book`                                    | 3    |
| Suggest the earliest opening that fits           | `next_free_slot`                              | 3    |

**Keys and values.** The key is the meeting's start time, stored as
*minutes since midnight*: 13:45 is `825`. Plain ints compare and add
easily, so a 15-minute meeting at `825` ends at `840`. The setup cell
gives you two helpers, `to_minutes("13:45") == 825` and
`fmt(825) == "13:45"`, plus `OPEN = 780` (13:00) and `CLOSE = 1020`
(17:00). The value is a `(name, minutes)` tuple, e.g.
`("Chloe Swift", 20)`.

**Why a BST and not last week's HashMap?** A hash table is great at one
question, "is this exact key here?", in O(1). But look at the table
above: most of those questions are about keys that are *not* in the map.
Nobody booked exactly 15:05, yet "who is in the room at 15:05" still has
an answer. A hash table scatters keys on purpose, so it has no idea which
key is *closest* to 15:05 without checking every single one. A binary
search tree keeps its keys in order, which turns each of these questions
into one walk down the tree.

**One naming rule for this notebook:** store the root in `self._root` and
the count in `self._size` (matching the real assignment's convention), and
keep the provided `_Node` class with its fields `_key`, `_value`, `_left`,
`_right`, `_parent`. The "Tree internals" panel in the app reads these
directly to draw your tree and to count how many nodes each operation
looked at.

**A note on input validation:** a real booking system would also have to
reject a blank name or a negative length. This lab skips that on purpose,
to keep the focus on the tree itself: assume every value you're given is
valid.

## Part 1: The binary search tree

Implement `BSTMap` below. Every node holds one key and its value, and the
whole tree obeys one rule, the **BST property**: everything in a node's
left subtree is smaller than the node's key, and everything in its right
subtree is larger. So at every node, a single comparison tells you which
way to go, and you never have to look at the other side at all.

```
            14:30              Looking for 15:15?
           /     \               15:15 > 14:30, go right
       13:30     16:00           15:15 < 16:00, go left
                 /    \          found it, 3 nodes looked at
             15:15    16:45
```

A few things to keep in mind:

* **Inserting always adds a leaf.** `__setitem__` walks down exactly like
  a search; when the next step would be `None`, that empty spot is where
  the new node goes. If it meets the same key on the way, it just
  overwrites the value, and the size stays the same.
* **Deleting is the hard part.** A node with 0 or 1 children is easy to
  unhook: its child (or nothing) takes its place. A node with 2 children
  can't just be unhooked, so instead you copy its **in-order successor**
  (the smallest key in its right subtree) into it, then delete the
  successor's node, which is always one of the easy cases. The docstring
  walks through it.
* **Links go both ways.** Every node knows its `_parent`. Whenever a node
  moves, fix the parent's `_left`/`_right` *and* the child's `_parent`.
  The app checks both after every click.
* **Loops, not recursion.** Task 10 builds a tree that is one long chain,
  and deep recursion down a long chain is exactly what crashes Python.
  `_search`, `__setitem__` and `__iter__` should all use loops
  (`__iter__` with an explicit stack).

This mirrors the real assignment's `dsa/maps/bst_map.py` almost exactly.

In [2]:
from typing import TypeVar, Generic, Iterator, Optional

K = TypeVar('K')
V = TypeVar('V')


class BSTMap(Generic[K, V]):
    """A map implemented as an unbalanced binary search tree.

    BST property: for every node, every key in its LEFT subtree is
    smaller than the node's key, and every key in its RIGHT subtree is
    larger. Every method below uses that one rule to decide which way
    to walk.
    """

    class _Node:
        """One node of the tree. Provided, nothing to edit here."""
        __slots__ = '_key', '_value', '_left', '_right', '_parent'

        def __init__(self, key, value, parent=None):
            self._key = key
            self._value = value
            self._left = None
            self._right = None
            self._parent = parent

    def __init__(self):
        """Create an empty map.

        Store the root node in `self._root` (None while the tree is
        empty) and the number of entries in `self._size`, starting at 0.
        """
        # YOUR CODE HERE
        raise NotImplementedError

    def __len__(self) -> int:
        """Return the number of key-value pairs stored (self._size)."""
        # YOUR CODE HERE
        raise NotImplementedError

    def _search(self, key: K, node) -> Optional['BSTMap._Node']:
        """Walk down from `node` looking for `key`.

        Return the node whose key equals `key`, or None once you fall
        off the bottom of the tree. Use a loop, not recursion.
        """
        # YOUR CODE HERE
        raise NotImplementedError

    def __getitem__(self, key: K) -> V:
        """Return the value stored for `key`.

        Raises:
            KeyError: if `key` is not present.
        """
        # YOUR CODE HERE
        raise NotImplementedError

    def __contains__(self, key: K) -> bool:
        """Return True if `key` is currently stored."""
        # YOUR CODE HERE
        raise NotImplementedError

    def __setitem__(self, key: K, value: V) -> None:
        """Associate `value` with `key`.

        Walk down from the root the same way `_search` does:
          * if you reach a node with this exact key, overwrite its value
            (the size does NOT change);
          * otherwise, when the next step would be None, attach a new
            node there as a leaf. Pass `parent=` so the new node knows
            its parent, and add 1 to `self._size`.

        Use a loop, not recursion: Task 10 builds a tree that is one
        long chain.
        """
        # YOUR CODE HERE
        raise NotImplementedError

    def __delitem__(self, key: K) -> None:
        """Remove `key` and its value.

        Three cases:
          * 0 children: unhook the node from its parent.
          * 1 child: the child moves up and takes the node's place.
          * 2 children: copy the key and value of the node's in-order
            successor (the smallest key in its RIGHT subtree, see
            `_subtree_min`) into the node, then remove the successor's
            node instead. The successor never has a left child, so
            removing it is always case 0 or case 1.

        When a child moves up, fix BOTH directions of the link: the
        parent's `_left` or `_right` must point to the child, and the
        child's `_parent` must point to the parent. If the removed node
        was the root, update `self._root`.

        Raises:
            KeyError: if `key` is not present.
        """
        # YOUR CODE HERE
        raise NotImplementedError

    def __iter__(self) -> Iterator[K]:
        """Yield every key in increasing order (in-order traversal).

        Use an explicit stack instead of recursion: go left as far as
        you can, pushing every node on the way; pop one, yield its key,
        then continue the same way from its right child.
        """
        # YOUR CODE HERE
        raise NotImplementedError

    def _subtree_min(self, node) -> 'BSTMap._Node':
        """Return the node with the smallest key in the subtree rooted
        at `node`."""
        # YOUR CODE HERE
        raise NotImplementedError

    def _subtree_max(self, node) -> 'BSTMap._Node':
        """Return the node with the largest key in the subtree rooted
        at `node`."""
        # YOUR CODE HERE
        raise NotImplementedError

    def is_empty(self) -> bool:
        """Return True if the map contains no key-value pairs."""
        return len(self) == 0

## Part 2: Ordered queries, what a hash table can't do

Part 1's `_search` answers one question: "does a meeting start at exactly
15:05?". Almost always the answer is no, because nobody booked 15:05. But
the question the front desk actually asks is "**who is in the room at
15:05?**", which means "what is the *latest* meeting that started at or
before 15:05?". That is the **floor** of 15:05. Its twin, the
**ceiling**, is the *earliest* meeting starting at or after 15:05: who's
next.

Could you answer that with plain `in` instead? Sure: try 15:05, then
15:04, then 15:03, ... until something is booked. Task 7 has you measure
how wasteful that is, and asks what happens if times were stored to the
millisecond.

Here's the key insight. When `_search(15:05)` falls off the bottom of the
tree, it has **already walked right past both neighbors** of 15:05 on the
way down: it stopped at the exact spot where 15:05 *would* be. It just
throws that information away and returns `None`. `floor_key` and
`ceiling_key` walk the very same path, but **remember the best candidate
seen so far** instead of throwing it away. Same walk, same O(h) cost, a
much more useful answer.

`range_keys(lo, hi)` is the third ordered query: every key between `lo`
and `hi`, in order. The point is to **skip whole subtrees** that can't
contain an answer: if a node's key is already smaller than `lo`, nothing
in its left subtree can be in range either, so don't even go there. The
app counts how many nodes your `range_keys` looks at (Task 8).

`OrderedBSTMap` *inherits* everything from your `BSTMap`, so you only add
the new methods. Remember to re-run this cell whenever you re-run Part 1.

In [3]:
class OrderedBSTMap(BSTMap[K, V]):
    """A BSTMap that can also answer "nearest key" questions.

    Part 1's `_search` can only answer "is this exact key here?". The
    methods below answer questions about keys that are NOT in the map,
    which is what a schedule really needs ("who is in the room at
    15:05?" when nobody booked exactly 15:05).
    """

    def floor_key(self, key: K) -> Optional[K]:
        """Return the largest stored key that is <= `key`, or None if
        every stored key is larger.

        Walk down from the root exactly like `_search`. Every time you
        step RIGHT, the node you are leaving is smaller than `key`, so
        it is a candidate answer: remember it. When you fall off the
        tree, the last candidate you remembered is the answer. One walk,
        O(h), no matter how many keys are stored.
        """
        # YOUR CODE HERE
        raise NotImplementedError

    def ceiling_key(self, key: K) -> Optional[K]:
        """Return the smallest stored key that is >= `key`, or None if
        every stored key is smaller.

        Mirror image of `floor_key`: remember the node every time you
        step LEFT.
        """
        # YOUR CODE HERE
        raise NotImplementedError

    def range_keys(self, lo: K, hi: K) -> Iterator[K]:
        """Yield every stored key k with lo <= k <= hi, in increasing
        order. Start the walk at the root (see `_range`)."""
        # YOUR CODE HERE
        raise NotImplementedError

    def _range(self, node, lo: K, hi: K) -> Iterator[K]:
        """Yield the keys in [lo, hi] from the subtree rooted at `node`,
        in increasing order, WITHOUT visiting the whole tree:

          * only go into the left subtree if lo < node's key
            (otherwise everything down there is too small);
          * yield the node's own key if lo <= key <= hi;
          * only go into the right subtree if node's key < hi
            (otherwise everything down there is too big).

        Recursion is fine here: `yield from self._range(child, lo, hi)`.
        """
        # YOUR CODE HERE
        raise NotImplementedError

## Part 3: The booking rules

Parts 1 and 2 built a data structure. Part 3 **uses** it. `can_book` and
`next_free_slot` are plain functions: they never touch `_root`, `_left`
or `_right`, only the map's public methods (`schedule.floor_key(t)`,
`schedule.ceiling_key(t)`, `schedule[t]`). That's the position you're in
whenever you use someone else's map: you know what it can do, not how
it's built.

Two things to get right:

* **Meetings are half-open intervals.** A meeting at 13:50 for 10 minutes
  covers `[13:50, 14:00)`: it's over *at* 14:00. So a meeting that ends
  at 14:00 and another that starts at 14:00 do **not** clash.
  Back-to-back is allowed.
* **Don't try every minute.** `can_book` needs exactly two lookups, the
  floor and the ceiling of the new start time (Task 9 asks you to explain
  why two are enough). `next_free_slot` should jump from one booking to
  the next with `ceiling_key`, not walk the clock minute by minute.

The app's **Book** button calls your `can_book` first and only inserts the
meeting if it says yes. Until you've written `can_book`, Book still works,
it just skips the clash check (and says so).

In [4]:
def can_book(schedule: OrderedBSTMap, start: int, duration: int) -> bool:
    """Return True if a meeting [start, start + duration) can be added.

    It can be added only if:
      * it fits inside office hours: OPEN <= start and
        start + duration <= CLOSE, and
      * it does not overlap any booking already in `schedule`.

    `schedule` maps start minute -> (name, duration). Meetings are
    half-open intervals: one that ends at 14:00 does NOT clash with one
    that starts at 14:00.

    You only need two lookups: `schedule.floor_key(start)` and
    `schedule.ceiling_key(start)`. Task 9 asks you why two are enough.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def next_free_slot(schedule: OrderedBSTMap, after: int,
                   duration: int) -> Optional[int]:
    """Return the earliest start time t >= `after` at which a meeting
    of `duration` minutes could be booked (inside office hours, no
    clash), or None if nothing fits before CLOSE.

    Don't try every minute. Jump from booking to booking instead:
      1. Start at t = max(after, OPEN).
      2. If the booking at floor_key(t) is still running at t, move t
         to the moment it ends.
      3. Look at nxt = ceiling_key(t). If there is no nxt, or the
         meeting would end by the time nxt starts, t is the answer (as
         long as t + duration <= CLOSE). Otherwise move t to the end of
         nxt and repeat step 3.
    """
    # YOUR CODE HERE
    raise NotImplementedError

## The booking app

Everything in this app, including every button, the **Load order**
switch, and the two panels, is provided code. You don't need to read,
touch, or understand any of it; it only exists to let you test whatever
you've implemented above by clicking, instead of writing your own test
code.

Fill in the fields and click a button: this works right now, before
you've written any code, it'll just say a method isn't implemented yet.
As you finish pieces above and re-run that cell, the matching buttons
start working. Type times like `13:45`.

* **Bookings** use Part 1 (**Book**, **Look Up**, **Cancel**, **Show
  Schedule**) and Part 3 (Book's clash check, **Next Free Slot**).
* **Ordered queries** use Part 2. **Range** goes from **Time** to
  **Until**. **Next ≥ using only in** is the brute-force alternative to
  **Ceiling ≥** for Task 7: it only ever asks `t in schedule`.
* **Load Sample Data** clears the schedule, then inserts
  `SAMPLE_BOOKINGS` either in **Booking order** (as listed) or
  **Chronological** (sorted by time). The switch only takes effect when
  you click Load.

Two panels below the buttons:
- **Action log**: what each button did, which nodes your code walked
  through, or which error it hit.
- **Tree internals**: a live drawing of your tree, read straight from
  `self._root`. Left to right is time order, top to bottom is depth.
  Nodes your code looked at during the last click (read their `_key`,
  `_left` or `_right`) are **blue, numbered in visit order**; the answer
  is **green**. The chips on top show the number of bookings, how many
  levels the tree has, the fewest levels that many nodes could possibly
  fit in, how many nodes the last action looked at, and whether your tree
  passed its checks (BST order, parent links, size).

In [5]:
# A fixed schedule everyone in the class uses, so results are comparable.
# Rows are listed in the order students signed up, which is NOT time
# order. Inserted in this order, the 12 bookings build a tree with the
# fewest levels 12 nodes can fit in. The "Load order" switch in the app
# can also insert the same rows sorted by time: watch what that does to
# the tree in Task 10.
SAMPLE_BOOKINGS = [
    # (start, name, minutes)
    ("14:30", "Frank Harris", 20),
    ("13:30", "Chloe Swift", 20),
    ("16:00", "Jack Bennett", 15),
    ("13:00", "Alice Copper", 15),
    ("13:50", "Daniel Lee", 10),
    ("15:15", "Henry Scott", 15),
    ("16:45", "Liam Foster", 15),
    ("13:15", "Ben Carter", 10),
    ("14:10", "Emma Wilson", 15),
    ("15:00", "Grace Turner", 10),
    ("15:40", "Ivy Parker", 20),
    ("16:30", "Kim Nguyen", 10),
]

In [6]:
# Everything in this cell is provided app code. You don't need to read,
# touch, or understand any of it: it only exists so you can test your
# OrderedBSTMap, can_book and next_free_slot by clicking.
import html as _html
import inspect as _inspect
import re as _re
from contextlib import contextmanager as _contextmanager

_APP_FILE = _inspect.currentframe().f_code.co_filename

# _app holds the live schedule plus what the last action did. The schedule
# is built lazily (on first click, not when this cell runs) so the UI
# below always renders, even before anything is implemented.
_app = {
    "instance": None,   # the live OrderedBSTMap
    "log": [],          # (kind, html) entries, oldest first
    "touched": [],      # id(node) for every node the last action looked at, in visit order
    "found": set(),     # keys to highlight as the answer of the last action
    "stale_warned": None,
}


# ---------------------------------------------------------------------
# Look and feel: pure decoration, no logic lives here.
# ---------------------------------------------------------------------
_CSS = """
<style>
.ohl-app {
  --ink: #1b1f3b; --ink-2: #555b7a; --ink-3: #8a90ad;
  --edge: rgba(255,255,255,.78); --glass: rgba(255,255,255,.44);
  --blue: #0a84ff; --green: #34c759; --orange: #ff9f0a; --red: #ff453a;
  --pink: #ff375f; --gray: #8e8e93;
  font-family: -apple-system, BlinkMacSystemFont, "SF Pro Text", "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
  color: var(--ink);
  border-radius: 30px;
  padding: 22px 22px 16px 22px !important;
  box-sizing: border-box;
  background:
    radial-gradient(40% 45% at 6% 8%, rgba(90,150,255,.55), transparent 70%),
    radial-gradient(32% 42% at 96% 6%, rgba(255,120,190,.42), transparent 70%),
    radial-gradient(42% 50% at 88% 96%, rgba(80,215,190,.42), transparent 70%),
    radial-gradient(36% 46% at 14% 98%, rgba(255,190,80,.38), transparent 70%),
    linear-gradient(160deg, #edf1ff 0%, #f5efff 50%, #ecf9f4 100%);
  box-shadow: 0 24px 70px rgba(30,40,100,.18);
}
.ohl-app .widget-box, .ohl-app .widget-hbox, .ohl-app .widget-vbox { overflow: visible !important; }
.ohl-app .widget-html, .ohl-app .widget-html-content {
  line-height: 1.45 !important; white-space: normal !important; overflow: visible !important;
  height: auto !important; font-family: inherit !important; color: var(--ink);
}

/* header */
.ohl-head { display: flex; align-items: center; gap: 14px; padding: 2px 4px 14px 4px; }
.ohl-icon {
  width: 48px; height: 48px; border-radius: 14px; flex: 0 0 auto;
  display: flex; align-items: center; justify-content: center;
  background: linear-gradient(160deg, #64d2ff 0%, #0a84ff 55%, #5e5ce6 100%);
  box-shadow: inset 0 1px 1px rgba(255,255,255,.75), 0 8px 18px rgba(10,132,255,.35);
}
.ohl-title { font-size: 26px; font-weight: 700; letter-spacing: -.4px; line-height: 1.1; }
.ohl-sub { font-size: 13px; color: var(--ink-2); margin-top: 3px; }
.ohl-group {
  font-size: 11px; font-weight: 700; letter-spacing: .7px; text-transform: uppercase;
  color: var(--ink-3); margin: 10px 0 0 8px;
}
.ohl-foot { font-size: 11px; color: var(--ink-3); text-align: right; margin-top: 10px; }

/* frosted glass bar holding the inputs */
.ohl-app .ohl-bar {
  background: var(--glass); border: 1px solid var(--edge); border-radius: 26px;
  padding: 7px 12px !important; margin: 0 0 4px 0;
  backdrop-filter: blur(22px) saturate(170%); -webkit-backdrop-filter: blur(22px) saturate(170%);
  box-shadow: inset 0 1px 0 rgba(255,255,255,.95), 0 8px 24px rgba(30,40,100,.08);
}
.ohl-app .widget-text, .ohl-app .widget-dropdown {
  height: auto !important; margin: 3px 10px 3px 0 !important; align-items: center;
}
.ohl-app .widget-text input, .ohl-app .widget-dropdown select {
  height: 34px !important; border-radius: 999px !important;
  border: 1px solid rgba(255,255,255,.95) !important;
  background-color: rgba(255,255,255,.62) !important;
  box-shadow: inset 0 1px 2px rgba(20,30,80,.10) !important;
  color: var(--ink) !important; font-size: 13px !important; font-family: inherit !important;
  padding: 0 14px !important; transition: box-shadow .2s ease, background-color .2s ease;
}
.ohl-app .widget-dropdown select { padding-right: 30px !important; }
.ohl-app .widget-text input:focus, .ohl-app .widget-dropdown select:focus {
  outline: none !important; background-color: rgba(255,255,255,.88) !important;
  box-shadow: 0 0 0 3px rgba(10,132,255,.28), inset 0 1px 2px rgba(20,30,80,.08) !important;
}
.ohl-app .widget-label {
  color: var(--ink-2) !important; font-weight: 600 !important; font-size: 12px !important;
  font-family: inherit !important; padding-right: 2px;
}

/* liquid glass pill buttons */
.ohl-app button.ohl-btn {
  --tint: rgba(255,255,255,.30); --tint-ink: #1b1f3b;
  position: relative; overflow: hidden;
  width: auto !important; height: 38px !important;
  padding: 0 18px !important; margin: 5px 8px 5px 0 !important;
  border-radius: 999px !important;
  border: 1px solid rgba(255,255,255,.85) !important;
  background:
    radial-gradient(120% 75% at 50% -12%, rgba(255,255,255,.92) 0%, rgba(255,255,255,.30) 46%, rgba(255,255,255,0) 62%),
    radial-gradient(90% 60% at 50% 125%, rgba(255,255,255,.55) 0%, rgba(255,255,255,0) 60%),
    var(--tint) !important;
  backdrop-filter: blur(16px) saturate(190%); -webkit-backdrop-filter: blur(16px) saturate(190%);
  box-shadow:
    inset 0 1px 1px rgba(255,255,255,.95),
    inset 0 -1px 1px rgba(255,255,255,.40),
    inset 0 0 0 1px rgba(255,255,255,.22),
    0 1px 2px rgba(20,30,80,.10),
    0 8px 20px -6px rgba(20,30,80,.28) !important;
  color: var(--tint-ink) !important;
  font-family: inherit !important; font-size: 13px !important; font-weight: 600 !important;
  letter-spacing: .1px; text-shadow: 0 1px 0 rgba(255,255,255,.55);
  transition: transform .28s cubic-bezier(.2,.9,.3,1.35), box-shadow .28s ease, filter .28s ease;
}
.ohl-app button.ohl-btn::after {
  content: ""; position: absolute; top: 0; left: -70%; width: 45%; height: 100%;
  background: linear-gradient(100deg, rgba(255,255,255,0), rgba(255,255,255,.65), rgba(255,255,255,0));
  transform: skewX(-20deg); opacity: 0; pointer-events: none;
  transition: left .7s ease, opacity .3s ease;
}
.ohl-app button.ohl-btn:hover {
  transform: translateY(-1px) scale(1.04);
  filter: saturate(1.3) brightness(1.04);
  box-shadow:
    inset 0 1px 1px rgba(255,255,255,1),
    inset 0 -1px 1px rgba(255,255,255,.45),
    inset 0 0 0 1px rgba(255,255,255,.30),
    0 2px 4px rgba(20,30,80,.10),
    0 14px 28px -8px rgba(20,30,80,.34) !important;
}
.ohl-app button.ohl-btn:hover::after { left: 125%; opacity: 1; }
.ohl-app button.ohl-btn:active { transform: scale(.95); transition-duration: .08s; }
.ohl-app button.ohl-btn:focus { outline: none !important; }
.ohl-app button.ohl-btn:focus-visible {
  box-shadow: 0 0 0 3px rgba(10,132,255,.35), inset 0 1px 1px rgba(255,255,255,.95) !important;
}
.ohl-app button.tint-green  { --tint: rgba(52,199,89,.30);  --tint-ink: #0f5a26; }
.ohl-app button.tint-blue   { --tint: rgba(10,132,255,.26); --tint-ink: #0a3f86; }
.ohl-app button.tint-red    { --tint: rgba(255,69,58,.24);  --tint-ink: #8a1610; }
.ohl-app button.tint-indigo { --tint: rgba(94,92,230,.24);  --tint-ink: #2d2a8a; }
.ohl-app button.tint-teal   { --tint: rgba(48,176,199,.26); --tint-ink: #0b5563; }
.ohl-app button.tint-orange { --tint: rgba(255,159,10,.28); --tint-ink: #7a4300; }
.ohl-app button.tint-clear  { --tint: rgba(255,255,255,.34); --tint-ink: #1b1f3b; }

/* segmented control (Load order) */
.ohl-app .ohl-seg {
  width: auto !important; height: auto !important; margin: 5px 10px 5px 0 !important;
  padding: 3px !important; border-radius: 999px;
  background: rgba(120,130,175,.16);
  box-shadow: inset 0 1px 2px rgba(20,30,80,.12), 0 1px 0 rgba(255,255,255,.8);
}
.ohl-app .ohl-seg button.widget-toggle-button {
  width: auto !important; height: 32px !important; margin: 0 !important;
  padding: 0 16px !important; border: 0 !important; border-radius: 999px !important;
  background: transparent !important; box-shadow: none !important;
  color: var(--ink-2) !important; font-family: inherit !important;
  font-size: 12.5px !important; font-weight: 600 !important;
  transition: background .25s ease, box-shadow .25s ease, color .25s ease;
}
.ohl-app .ohl-seg button.widget-toggle-button.mod-active {
  background: linear-gradient(180deg, rgba(255,255,255,.98), rgba(255,255,255,.74)) !important;
  color: var(--ink) !important;
  box-shadow: inset 0 1px 1px rgba(255,255,255,1), 0 4px 12px rgba(20,30,80,.18) !important;
}

/* frosted glass cards: action log + tree */
.ohl-card {
  background: var(--glass); border: 1px solid var(--edge); border-radius: 24px;
  padding: 14px 16px; margin-top: 12px;
  backdrop-filter: blur(24px) saturate(170%); -webkit-backdrop-filter: blur(24px) saturate(170%);
  box-shadow: inset 0 1px 0 rgba(255,255,255,.95), 0 10px 30px rgba(30,40,100,.10);
}
.ohl-card-title { font-size: 15px; font-weight: 700; margin: 0 0 10px 2px; }
.ohl-empty { color: var(--ink-3); font-size: 13px; padding: 2px; }
.ohl-card code {
  font-family: "SF Mono", Menlo, Consolas, monospace; font-size: 12px;
  background: rgba(30,40,100,.07); padding: 1px 6px; border-radius: 7px;
}
.ohl-mono { font-family: "SF Mono", Menlo, Consolas, monospace; font-size: 12.5px; }
.ohl-log { display: flex; flex-direction: column; gap: 6px; }
.ohl-entry {
  display: flex; gap: 10px; align-items: flex-start; padding: 8px 12px;
  border-radius: 16px; background: rgba(255,255,255,.58);
  border: 1px solid rgba(255,255,255,.9); font-size: 13px; line-height: 1.45;
}
.ohl-dot {
  flex: 0 0 auto; width: 20px; height: 20px; border-radius: 50%; margin-top: 1px;
  display: flex; align-items: center; justify-content: center;
  color: #fff; font-size: 11px; font-weight: 800;
  box-shadow: inset 0 1px 1px rgba(255,255,255,.6);
}
.ohl-entry.ok .ohl-dot   { background: var(--green); }
.ohl-entry.info .ohl-dot { background: var(--blue); }
.ohl-entry.no .ohl-dot   { background: var(--pink); }
.ohl-entry.warn .ohl-dot { background: var(--orange); }
.ohl-entry.err .ohl-dot  { background: var(--red); }
.ohl-entry.todo .ohl-dot { background: var(--gray); }
.ohl-entry.warn { background: rgba(255,159,10,.12); border-color: rgba(255,159,10,.30); }
.ohl-entry.err  { background: rgba(255,69,58,.10);  border-color: rgba(255,69,58,.28); }
.ohl-rows { display: grid; grid-template-columns: auto 1fr; gap: 2px 14px; margin-top: 4px; }
.ohl-chips { display: flex; flex-wrap: wrap; gap: 6px; margin-bottom: 10px; }
.ohl-chip {
  font-size: 12px; font-weight: 600; padding: 4px 11px; border-radius: 999px;
  background: rgba(255,255,255,.62); border: 1px solid rgba(255,255,255,.95);
  box-shadow: inset 0 1px 0 rgba(255,255,255,.9); color: var(--ink-2);
}
.ohl-chip b { color: var(--ink); }
.ohl-chip.blue   { background: rgba(10,132,255,.13); border-color: rgba(10,132,255,.25); color: #0a3f86; }
.ohl-chip.green  { background: rgba(52,199,89,.15);  border-color: rgba(52,199,89,.30);  color: #0f5a26; }
.ohl-chip.orange { background: rgba(255,159,10,.16); border-color: rgba(255,159,10,.32); color: #7a4300; }
.ohl-issues {
  margin: 0 0 10px 0; padding: 8px 12px; border-radius: 14px; font-size: 12.5px;
  background: rgba(255,159,10,.13); border: 1px solid rgba(255,159,10,.35); color: #7a4300;
}
.ohl-tree { overflow-x: auto; padding: 2px 0 4px 0; }
.ohl-legend { display: flex; flex-wrap: wrap; gap: 16px; font-size: 11.5px; color: var(--ink-2); margin-top: 6px; }
.ohl-swatch {
  display: inline-block; width: 12px; height: 12px; border-radius: 50%;
  margin-right: 6px; vertical-align: -2px;
}
</style>
"""

_HEADER = """
<div class="ohl-head">
  <div class="ohl-icon">
    <svg width="26" height="26" viewBox="0 0 24 24" fill="none" stroke="#fff"
         stroke-width="2" stroke-linecap="round" stroke-linejoin="round">
      <rect x="3" y="5" width="18" height="16" rx="3"></rect>
      <path d="M3 10h18M8 3v4M16 3v4"></path>
      <path d="M12 13.5v3l2 1.2"></path>
    </svg>
  </div>
  <div>
    <div class="ohl-title">Office Hours</div>
    <div class="ohl-sub">CSCI 232 &middot; Tuesdays 13:00&ndash;17:00 &middot;
      runs on the <code>OrderedBSTMap</code> you write above</div>
  </div>
</div>
"""

_FONT = "-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif"


def _esc(value):
    return _html.escape(str(value))


def _lbl(key):
    """How a key is shown: 825 -> '13:45'."""
    if isinstance(key, int) and not isinstance(key, bool) and 0 <= key < 24 * 60:
        return fmt(key)
    return _esc(repr(key))


def _unpack(value):
    """A booking value is (name, minutes); cope with anything else."""
    try:
        name, minutes = value
        return str(name), int(minutes)
    except Exception:
        return str(value), 0


def _span(start, minutes):
    return f"{fmt(start)}&ndash;{fmt(start + minutes)}"


def _plural(n, word):
    return f"{n} {word}{'' if n == 1 else 's'}"


# ---------------------------------------------------------------------
# Reading the student's tree without trusting it, and watching which
# nodes their code looks at.
# ---------------------------------------------------------------------
_MAX_NODES = 300


def _snapshot(inst):
    """In-order list of (node, depth), plus any structural problems.
    Never loops forever, even if a bad link points back up the tree."""
    nodes, issues, seen = [], [], set()
    stack = []
    node, depth = inst._root, 0
    while stack or node is not None:
        while node is not None:
            if id(node) in seen:
                issues.append("a node is reachable twice: some link points back up the tree")
                node = None
                break
            if len(seen) >= _MAX_NODES:
                issues.append(f"stopped reading after {_MAX_NODES} nodes")
                return nodes, issues
            seen.add(id(node))
            stack.append((node, depth))
            node, depth = node._left, depth + 1
        if not stack:
            break
        node, depth = stack.pop()
        nodes.append((node, depth))
        node, depth = node._right, depth + 1
    return nodes, issues


def _integrity(inst, nodes):
    issues = []
    keys = [n._key for n, _ in nodes]
    if any(not (a < b) for a, b in zip(keys, keys[1:])):
        issues.append("keys are not in BST order (left &lt; node &lt; right is broken somewhere)")
    root = inst._root
    if root is not None and getattr(root, "_parent", None) is not None:
        issues.append(f"the root ({_lbl(root._key)}) has a <code>_parent</code>, it should be None")
    for n, _ in nodes:
        for child in (n._left, n._right):
            if child is not None and getattr(child, "_parent", None) is not n:
                issues.append(f"<code>_parent</code> of {_lbl(child._key)} doesn't point to {_lbl(n._key)}")
    if len(nodes) != inst._size:
        issues.append(f"<code>_size</code> is {_esc(inst._size)} but the tree has {len(nodes)} nodes")
    return issues


def _entries(inst):
    """{key: value} read straight from the nodes (not through your methods)."""
    nodes, _ = _snapshot(inst)
    return {n._key: n._value for n, _ in nodes}


def _preorder(inst):
    """(key, value) pairs root-first: re-inserting them in this order
    rebuilds exactly the same tree shape."""
    out, seen, stack = [], set(), [inst._root]
    while stack:
        node = stack.pop()
        if node is None or id(node) in seen or len(seen) >= _MAX_NODES:
            continue
        seen.add(id(node))
        out.append((node._key, node._value))
        stack.append(node._right)
        stack.append(node._left)
    return out


class _Visits:
    """Every node the student's code looked at during one action, in
    visit order. A node counts as looked at when the code reads its
    `_key`, `_left` or `_right`."""

    def __init__(self, key_slot):
        self.key_slot = key_slot
        self.order = {}    # id(node) -> its key when first visited

    def note(self, node):
        if id(node) not in self.order:
            try:
                self.order[id(node)] = self.key_slot.__get__(node, type(node))
            except AttributeError:
                self.order[id(node)] = "?"


class _SlotSpy:
    """Stands in for one _Node slot (_key, _left or _right) while an
    action runs, reporting every read to a _Visits."""

    def __init__(self, slot, visits):
        self.slot = slot
        self.visits = visits

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        self.visits.note(obj)
        return self.slot.__get__(obj, objtype)

    def __set__(self, obj, value):
        self.slot.__set__(obj, value)


def _slot(node_cls, name):
    for klass in getattr(node_cls, "__mro__", ()):
        candidate = klass.__dict__.get(name)
        if candidate is not None:
            return (klass, candidate) if type(candidate).__name__ == "member_descriptor" else None
    return None


@_contextmanager
def _watch(inst):
    node_cls = getattr(type(inst), "_Node", None)
    slots = {name: _slot(node_cls, name) for name in ("_key", "_left", "_right")}
    if slots["_key"] is None:
        yield None
        return
    visits = _Visits(slots["_key"][1])
    patched = []
    try:
        for found in slots.values():
            if found is not None:
                owner, slot = found
                setattr(owner, slot.__name__, _SlotSpy(slot, visits))
                patched.append((owner, slot))
        yield visits
    finally:
        for owner, slot in patched:
            setattr(owner, slot.__name__, slot)


def _remember(spy, found=()):
    _app["touched"] = list(spy.order) if spy is not None else []
    _app["found"] = {k for k in found if k is not None}


def _path(spy):
    if spy is None:
        return "(visit tracking unavailable)"
    keys = list(spy.order.values())
    if not keys:
        return "no nodes"
    if len(keys) > 14:
        shown = " &rarr; ".join(_lbl(k) for k in keys[:12]) + f" &rarr; &hellip; +{len(keys) - 12}"
    else:
        shown = " &rarr; ".join(_lbl(k) for k in keys)
    return f"<span class='ohl-mono'>{shown}</span> ({_plural(len(keys), 'node')})"


# ---------------------------------------------------------------------
# Rendering the two panels.
# ---------------------------------------------------------------------
_GLYPH = {"ok": "&#10003;", "info": "i", "no": "&#10005;", "warn": "!", "err": "&#10005;", "todo": "&hellip;"}


def _log(kind, message):
    _app["log"].append((kind, message))


def _render_log():
    if not _app["log"]:
        body = '<div class="ohl-empty">Nothing yet: click a button above.</div>'
    else:
        body = '<div class="ohl-log">' + "".join(
            f'<div class="ohl-entry {kind}"><span class="ohl-dot">{_GLYPH[kind]}</span><div>{msg}</div></div>'
            for kind, msg in _app["log"]) + "</div>"
    return f'<div class="ohl-card"><div class="ohl-card-title">Action log</div>{body}</div>'


_COL, _ROW, _W, _H = 66, 62, 58, 34


def _tree_svg(nodes, touched, found):
    order = {nid: i + 1 for i, nid in enumerate(touched)}
    depth_max = max(d for _, d in nodes)
    width = len(nodes) * _COL + 16
    height = (depth_max + 1) * _ROW + 6
    pos = {id(n): (8 + _COL / 2 + i * _COL, 14 + _H / 2 + d * _ROW) for i, (n, d) in enumerate(nodes)}

    parts = [
        f'<svg width="{width}" height="{height}" viewBox="0 0 {width} {height}" '
        f'xmlns="http://www.w3.org/2000/svg" font-family="{_FONT}">',
        '<defs>'
        '<linearGradient id="ohl-g-plain" x1="0" y1="0" x2="0" y2="1">'
        '<stop offset="0" stop-color="#ffffff" stop-opacity=".97"/>'
        '<stop offset="1" stop-color="#ffffff" stop-opacity=".62"/></linearGradient>'
        '<linearGradient id="ohl-g-hot" x1="0" y1="0" x2="0" y2="1">'
        '<stop offset="0" stop-color="#eaf3ff"/><stop offset="1" stop-color="#bcdcff"/></linearGradient>'
        '<linearGradient id="ohl-g-found" x1="0" y1="0" x2="0" y2="1">'
        '<stop offset="0" stop-color="#ebfbef"/><stop offset="1" stop-color="#b7efc6"/></linearGradient>'
        '<filter id="ohl-shadow" x="-30%" y="-30%" width="160%" height="190%">'
        '<feDropShadow dx="0" dy="2" stdDeviation="2.4" flood-color="#1e2864" flood-opacity=".16"/>'
        '</filter></defs>',
    ]
    for n, _ in nodes:
        x, y = pos[id(n)]
        for child in (n._left, n._right):
            if child is not None and id(child) in pos:
                cx, cy = pos[id(child)]
                hot = id(n) in order and id(child) in order
                parts.append(
                    f'<line x1="{x:.1f}" y1="{y + _H / 2:.1f}" x2="{cx:.1f}" y2="{cy - _H / 2:.1f}" '
                    f'stroke="{"#0a84ff" if hot else "#46508a"}" stroke-opacity="{".8" if hot else ".32"}" '
                    f'stroke-width="{2.4 if hot else 1.5}" stroke-linecap="round"/>')
    for n, _ in nodes:
        x, y = pos[id(n)]
        name = _unpack(n._value)[0].split(" ")[0][:9]
        if n._key in found:
            fill, stroke, sw = "url(#ohl-g-found)", "#34c759", 2
        elif id(n) in order:
            fill, stroke, sw = "url(#ohl-g-hot)", "#0a84ff", 1.6
        else:
            fill, stroke, sw = "url(#ohl-g-plain)", "#ffffff", 1
        parts.append(
            f'<g><title>{_lbl(n._key)} {_esc(_unpack(n._value)[0])}</title>'
            f'<rect x="{x - _W / 2:.1f}" y="{y - _H / 2:.1f}" width="{_W}" height="{_H}" rx="{_H / 2}" '
            f'fill="{fill}" stroke="{stroke}" stroke-width="{sw}" filter="url(#ohl-shadow)"/>'
            f'<text x="{x:.1f}" y="{y - 1.5:.1f}" text-anchor="middle" font-size="12.5" '
            f'font-weight="700" fill="#1b1f3b">{_lbl(n._key)}</text>'
            f'<text x="{x:.1f}" y="{y + 11:.1f}" text-anchor="middle" font-size="9.5" '
            f'fill="#555b7a">{_esc(name)}</text></g>')
        if id(n) in order:
            bx, by = x + _W / 2 - 3, y - _H / 2 + 1
            parts.append(
                f'<circle cx="{bx:.1f}" cy="{by:.1f}" r="8.5" fill="#0a84ff" stroke="#ffffff" stroke-width="1.5"/>'
                f'<text x="{bx:.1f}" y="{by + 3.3:.1f}" text-anchor="middle" font-size="9.5" '
                f'font-weight="700" fill="#ffffff">{order[id(n)]}</text>')
    parts.append("</svg>")
    return "".join(parts)


def _render_tree():
    title = '<div class="ohl-card-title">Tree internals</div>'
    inst = _app["instance"]
    if inst is None:
        return f'<div class="ohl-card">{title}<div class="ohl-empty">No schedule yet: click any button to create one.</div></div>'
    try:
        inst._root, inst._size
    except AttributeError:
        return (f'<div class="ohl-card">{title}<div class="ohl-empty">Internal state not available yet: '
                'make sure <code>__init__</code> sets <code>self._root</code> and <code>self._size</code>.</div></div>')
    try:
        nodes, issues = _snapshot(inst)
        issues += _integrity(inst, nodes)
        n = len(nodes)
        levels = (max(d for _, d in nodes) + 1) if nodes else 0
        alive = {id(x) for x, _ in nodes}
        touched = [nid for nid in _app["touched"] if nid in alive]
        chips = [
            f'<span class="ohl-chip"><b>{n}</b> {"booking" if n == 1 else "bookings"}</span>',
            f'<span class="ohl-chip"><b>{levels}</b> {"level" if levels == 1 else "levels"}</span>',
            f'<span class="ohl-chip">fewest possible for {n}: <b>{n.bit_length()}</b></span>',
        ]
        if _app["touched"]:
            chips.append(f'<span class="ohl-chip blue">last action looked at <b>{len(_app["touched"])}</b> '
                         f'{"node" if len(_app["touched"]) == 1 else "different nodes"}</span>')
        if not issues:
            chips.append('<span class="ohl-chip green">&#10003; BST order &middot; parent links &middot; size</span>')
        warn = ""
        if issues:
            shown = issues[:4] + ([f"&hellip; and {len(issues) - 4} more"] if len(issues) > 4 else [])
            warn = '<div class="ohl-issues"><b>Check your tree:</b> ' + "; ".join(shown) + ".</div>"
        if nodes:
            drawing = f'<div class="ohl-tree">{_tree_svg(nodes, touched, _app["found"])}</div>'
            legend = ('<div class="ohl-legend">'
                      '<span><span class="ohl-swatch" style="background:#0a84ff"></span>'
                      'looked at by the last action (number = visit order)</span>'
                      '<span><span class="ohl-swatch" style="background:#34c759"></span>the answer</span>'
                      '<span>left to right = time order &middot; top to bottom = depth</span></div>')
        else:
            drawing, legend = '<div class="ohl-empty">The tree is empty (<code>_root</code> is None).</div>', ""
        return f'<div class="ohl-card">{title}<div class="ohl-chips">{"".join(chips)}</div>{warn}{drawing}{legend}</div>'
    except Exception as e:
        return (f'<div class="ohl-card">{title}<div class="ohl-issues">Couldn\'t draw the tree: '
                f'{type(e).__name__}: {_esc(e)}</div></div>')


def _refresh():
    log_view.value = _render_log()
    tree_view.value = _render_tree()


# ---------------------------------------------------------------------
# The schedule itself, and turning errors into friendly log lines.
# ---------------------------------------------------------------------
class _BadInput(Exception):
    pass


_TIME_RE = _re.compile(r"^\s*(\d{1,2})\s*[:.hH]?\s*(\d{2})\s*$")


def _read_time(box, label):
    match = _TIME_RE.match(box.value or "")
    if not match or int(match.group(1)) > 23 or int(match.group(2)) > 59:
        raise _BadInput(f"{label}: type a time like 13:45.")
    return int(match.group(1)) * 60 + int(match.group(2))


def _schedule():
    if BSTMap not in OrderedBSTMap.__mro__ and _app["stale_warned"] is not OrderedBSTMap:
        _app["stale_warned"] = OrderedBSTMap
        _log("warn", "You re-ran the Part 1 cell but not the Part 2 cell, so <code>OrderedBSTMap</code> "
                     "is still built on your <i>old</i> <code>BSTMap</code>. Re-run the Part 2 cell too.")
    inst = _app["instance"]
    if inst is not None and type(inst) is not OrderedBSTMap:
        rows = _preorder(inst)
        fresh = OrderedBSTMap()
        for key, value in rows:
            fresh[key] = value
        _app["instance"] = inst = fresh
        _log("info", "You re-ran a cell, so the schedule was rebuilt with your new code "
                     "(same bookings, same tree shape).")
    if inst is None:
        inst = OrderedBSTMap()
        _app["instance"] = inst
    return inst


def _where(exc):
    """Name of the student function an exception came from."""
    codes = []
    tb = exc.__traceback__
    while tb is not None:
        codes.append(tb.tb_frame.f_code)
        tb = tb.tb_next
    for code in reversed(codes):
        if code.co_filename != _APP_FILE and "contextlib" not in code.co_filename:
            return getattr(code, "co_qualname", code.co_name)
    return getattr(codes[-1], "co_qualname", codes[-1].co_name) if codes else "?"


def _guarded(handler):
    """Run a handler, turning unfinished methods / bad input / crashes
    into a log line instead of a traceback, then redraw both panels."""
    def run(_btn=None):
        _app["touched"], _app["found"] = [], set()
        try:
            handler()
        except _BadInput as e:
            _log("warn", _esc(e))
        except NotImplementedError as e:
            _log("todo", f"<code>{_esc(_where(e))}</code> isn't implemented yet: finish it above, "
                         "re-run that cell, then try again.")
        except Exception as e:
            _log("err", f"<code>{_esc(_where(e))}</code> raised <code>{type(e).__name__}: {_esc(e)}</code>")
        finally:
            _refresh()
    return run


def _conflict(entries, start, minutes):
    """The app's own clash check, used to double-check can_book."""
    if start < OPEN or start + minutes > CLOSE:
        return f"office hours are {fmt(OPEN)}&ndash;{fmt(CLOSE)}"
    for key in sorted(entries):
        name, length = _unpack(entries[key])
        if key < start + minutes and start < key + length:
            return f"it overlaps {_esc(name)} ({_span(key, length)})"
    return None


def _expected_free(entries, after, minutes):
    for t in range(max(after, OPEN), CLOSE - minutes + 1):
        if _conflict(entries, t, minutes) is None:
            return t
    return None


# ---------------------------------------------------------------------
# Button handlers.
# ---------------------------------------------------------------------
def _on_book():
    start = _read_time(time_box, "Time")
    name = name_box.value.strip()
    if not name:
        raise _BadInput("Type a name first.")
    minutes = duration_box.value
    sched = _schedule()
    entries = _entries(sched)
    clash = _conflict(entries, start, minutes)
    try:
        allowed = can_book(sched, start, minutes)
    except NotImplementedError:
        allowed = None
        _log("todo", "<code>can_book</code> isn't implemented yet, so this booking skips the clash check.")
    if allowed is False:
        if clash:
            _log("no", f"Can't book {_span(start, minutes)}: {clash}.")
        else:
            _log("warn", f"<code>can_book</code> said no, but {_span(start, minutes)} looks free. "
                         "Check your boundaries: a meeting that ends at 14:00 doesn't clash with one that starts at 14:00.")
        return
    if allowed is True and clash:
        _log("warn", f"<code>can_book</code> said yes, but {clash}. Not booked: check your logic.")
        return
    with _watch(sched) as spy:
        sched[start] = (name, minutes)
    _remember(spy, [start])
    verb = "Updated" if start in entries else "Booked"
    _log("ok", f"{verb} {_span(start, minutes)} for <b>{_esc(name)}</b>. Insert walked {_path(spy)}.")


def _on_lookup():
    t = _read_time(time_box, "Time")
    sched = _schedule()
    entries = _entries(sched)
    with _watch(sched) as spy:
        try:
            value, hit = sched[t], True
        except KeyError:
            value, hit = None, False
    _remember(spy, [t] if hit else [])
    if hit:
        name, minutes = _unpack(value)
        _log("ok", f"{fmt(t)} &rarr; <b>{_esc(name)}</b>, {minutes} min ({_span(t, minutes)}). "
                   f"Search walked {_path(spy)}.")
        if t not in entries:
            _log("warn", f"Hmm: no node in the tree has key {fmt(t)}. Check <code>__getitem__</code>.")
    else:
        _log("info", f"No booking <b>starts</b> at {fmt(t)} (<code>KeyError</code>). Search walked {_path(spy)}.")
        if t in entries:
            _log("warn", f"Hmm: the tree does have a node with key {fmt(t)}. Check <code>_search</code>.")


def _on_cancel():
    t = _read_time(time_box, "Time")
    sched = _schedule()
    entries = _entries(sched)
    with _watch(sched) as spy:
        try:
            del sched[t]
            removed = True
        except KeyError:
            removed = False
    _remember(spy)
    if removed:
        name = _unpack(entries[t])[0] if t in entries else "?"
        _log("ok", f"Cancelled {fmt(t)} ({_esc(name)}). Delete looked at {_path(spy)}.")
    else:
        _log("info", f"Nothing to cancel: no booking starts at {fmt(t)} (<code>KeyError</code>).")


def _on_schedule():
    sched = _schedule()
    entries = _entries(sched)
    with _watch(sched) as spy:
        keys = list(sched)
    _remember(spy)
    if not keys:
        _log("info", "The schedule is empty.")
        return
    rows = "".join(
        f"<span class='ohl-mono'>{_span(k, _unpack(entries.get(k))[1])}</span>"
        f"<span>{_esc(_unpack(entries.get(k))[0])}</span>" for k in keys)
    _log("ok", f"{_plural(len(keys), 'booking')}, in the order <code>__iter__</code> produced them:"
               f"<div class='ohl-rows'>{rows}</div>")
    if keys != sorted(entries):
        _log("warn", "Hmm: that isn't every key in increasing order. Check <code>__iter__</code>.")


def _on_floor():
    t = _read_time(time_box, "Time")
    sched = _schedule()
    entries = _entries(sched)
    with _watch(sched) as spy:
        key = sched.floor_key(t)
    _remember(spy, [key])
    expected = max((k for k in entries if k <= t), default=None)
    if key is None:
        msg = f"<code>floor_key({fmt(t)})</code> &rarr; None: no booking starts at or before {fmt(t)}."
    else:
        name, minutes = _unpack(entries.get(key, ("?", 0)))
        msg = f"<code>floor_key({fmt(t)})</code> &rarr; <b>{_lbl(key)}</b>. "
        if key + minutes > t:
            msg += f"{_esc(name)} is in the room until {fmt(key + minutes)}."
        else:
            msg += f"{_esc(name)}'s meeting ended at {fmt(key + minutes)}, so the room is free at {fmt(t)}."
    msg += f" Looked at {_path(spy)}."
    if key != expected:
        _log("warn", msg + f" Hmm: the tree says the answer should be "
                           f"{'None' if expected is None else fmt(expected)}.")
    else:
        _log("ok", msg)


def _on_ceiling():
    t = _read_time(time_box, "Time")
    sched = _schedule()
    entries = _entries(sched)
    with _watch(sched) as spy:
        key = sched.ceiling_key(t)
    _remember(spy, [key])
    expected = min((k for k in entries if k >= t), default=None)
    if key is None:
        msg = f"<code>ceiling_key({fmt(t)})</code> &rarr; None: no booking starts at or after {fmt(t)}."
    else:
        name, _ = _unpack(entries.get(key, ("?", 0)))
        msg = (f"<code>ceiling_key({fmt(t)})</code> &rarr; <b>{_lbl(key)}</b>: "
               f"next up is {_esc(name)} at {_lbl(key)}.")
    msg += f" Looked at {_path(spy)}."
    if key != expected:
        _log("warn", msg + f" Hmm: the tree says the answer should be "
                           f"{'None' if expected is None else fmt(expected)}.")
    else:
        _log("ok", msg)


def _on_brute():
    t0 = _read_time(time_box, "Time")
    sched = _schedule()
    calls = total = 0
    union = {}
    found = None
    for t in range(t0, 24 * 60):
        with _watch(sched) as spy:
            hit = t in sched
        calls += 1
        if spy is not None:
            total += len(spy.order)
            for nid, key in spy.order.items():
                union.setdefault(nid, key)
        if hit:
            found = t
            break
    _app["touched"] = list(union)
    _app["found"] = {found} if found is not None else set()
    if calls > 2:
        tried = f"{fmt(t0)}, {fmt(t0 + 1)}, &hellip;, {fmt(t0 + calls - 1)}"
    else:
        tried = ", ".join(fmt(t0 + i) for i in range(calls))
    if found is not None:
        _log("info", f"Using only <code>in</code>: asked <b>{calls}</b> times ({tried}) before a "
                     f"booking turned up at <b>{fmt(found)}</b>. Those {calls} searches looked at "
                     f"<b>{total}</b> nodes in total.")
    else:
        _log("info", f"Using only <code>in</code>: asked <b>{calls}</b> times, minute by minute up to "
                     f"midnight, and found nothing. Those searches looked at <b>{total}</b> nodes in total.")


def _on_range():
    lo = _read_time(time_box, "Time")
    hi = _read_time(until_box, "Until")
    sched = _schedule()
    entries = _entries(sched)
    with _watch(sched) as spy:
        keys = list(sched.range_keys(lo, hi))
    _remember(spy, keys)
    expected = [k for k in sorted(entries) if lo <= k <= hi]
    names = ", ".join(f"{_lbl(k)} {_esc(_unpack(entries.get(k, ('?', 0)))[0])}" for k in keys) or "nobody"
    seen = len(spy.order) if spy is not None else "?"
    msg = (f"<code>range_keys({fmt(lo)}, {fmt(hi)})</code> &rarr; {_plural(len(keys), 'booking')}: {names}. "
           f"Looked at <b>{seen}</b> of {len(entries)} nodes.")
    if keys != expected:
        _log("warn", msg + " Hmm: the tree says the answer should be "
                           f"[{', '.join(fmt(k) for k in expected)}].")
    else:
        _log("ok", msg)


def _on_free():
    after = _read_time(time_box, "Time")
    minutes = duration_box.value
    sched = _schedule()
    entries = _entries(sched)
    with _watch(sched) as spy:
        t = next_free_slot(sched, after, minutes)
    _remember(spy)
    expected = _expected_free(entries, after, minutes)
    if t is None:
        msg = f"<code>next_free_slot</code>: no {minutes}-minute opening between {fmt(max(after, OPEN))} and {fmt(CLOSE)}."
    else:
        msg = (f"<code>next_free_slot</code>: earliest {minutes}-minute opening at or after {fmt(after)} "
               f"&rarr; <b>{_lbl(t)}</b>.")
    if t != expected:
        _log("warn", msg + f" Hmm: the schedule says it should be "
                           f"{'None' if expected is None else fmt(expected)}.")
        return
    if t is not None:
        time_box.value = fmt(t)
        msg += f" Time is now set to {fmt(t)}: click <b>Book</b> to take it."
    _log("ok", msg)


def _on_load():
    rows = list(SAMPLE_BOOKINGS)
    if load_order.value == "sorted":
        rows.sort(key=lambda row: to_minutes(row[0]))
    _app["instance"] = None
    sched = _schedule()
    for hhmm, name, minutes in rows:
        sched[to_minutes(hhmm)] = (name, minutes)
    how = "time order (chronological)" if load_order.value == "sorted" else "booking order"
    _log("info", f"Loaded {len(rows)} sample bookings in <b>{how}</b> (the schedule was cleared first).")


def _on_reset():
    _app["instance"] = None
    _schedule()
    _log("info", "Schedule reset (fresh, empty <code>OrderedBSTMap</code>).")


def _on_order_change(change):
    how = "time order" if change["new"] == "sorted" else "booking order"
    _log("info", f"Load order set to <b>{how}</b>: click <b>Load Sample Data</b> to rebuild the tree that way.")
    _refresh()


def _on_clear_log(_btn=None):
    _app["log"] = []
    _refresh()


# ---------------------------------------------------------------------
# Widgets and layout.
# ---------------------------------------------------------------------
def _button(text, tint, tip):
    btn = widgets.Button(description=text, tooltip=tip, layout=widgets.Layout(width="auto"))
    btn.add_class("ohl-btn")
    btn.add_class(f"tint-{tint}")
    return btn


def _group(text):
    return widgets.HTML(f'<div class="ohl-group">{text}</div>', layout=widgets.Layout(width="100%"))


def _row(children):
    return widgets.HBox(children, layout=widgets.Layout(flex_flow="row wrap", align_items="center"))


_LABEL = {"description_width": "initial"}
time_box = widgets.Text(value="13:00", description="Time", placeholder="13:45",
                        layout=widgets.Layout(width="140px"), style=_LABEL)
name_box = widgets.Text(value="", description="Name", placeholder="Your name",
                        layout=widgets.Layout(width="230px"), style=_LABEL)
duration_box = widgets.Dropdown(options=[("10 min", 10), ("15 min", 15), ("20 min", 20), ("30 min", 30)],
                                value=15, description="Length", layout=widgets.Layout(width="150px"),
                                style=_LABEL)
until_box = widgets.Text(value="15:30", description="Until", placeholder="15:30",
                         layout=widgets.Layout(width="140px"), style=_LABEL)
inputs = _row([time_box, name_box, duration_box, until_box])
inputs.add_class("ohl-bar")

book_btn = _button("Book", "green", "can_book, then __setitem__")
lookup_btn = _button("Look Up", "blue", "__getitem__(Time)")
cancel_btn = _button("Cancel", "red", "__delitem__(Time)")
schedule_btn = _button("Show Schedule", "indigo", "__iter__")
free_btn = _button("Next Free Slot", "teal", "next_free_slot(Time, Length)")
floor_btn = _button("Floor ≤", "blue", "floor_key(Time)")
ceiling_btn = _button("Ceiling ≥", "blue", "ceiling_key(Time)")
brute_btn = _button("Next ≥ using only in", "orange",
                    "asks `Time in schedule`, then Time+1, Time+2, ... until one is booked")
range_btn = _button("Range", "indigo", "range_keys(Time, Until)")
load_btn = _button("Load Sample Data", "clear", "clear the schedule, then insert SAMPLE_BOOKINGS")
reset_btn = _button("Reset", "clear", "start from an empty OrderedBSTMap")
clear_log_btn = _button("Clear Log", "clear", "empty the action log")
load_order = widgets.ToggleButtons(options=[("Booking order", "booking"), ("Chronological", "sorted")],
                                   value="booking", style={"button_width": "auto"},
                                   tooltips=["insert the rows as listed", "insert the rows sorted by time"])
load_order.add_class("ohl-seg")

log_view = widgets.HTML(layout=widgets.Layout(width="100%"))
tree_view = widgets.HTML(layout=widgets.Layout(width="100%"))

book_btn.on_click(_guarded(_on_book))
lookup_btn.on_click(_guarded(_on_lookup))
cancel_btn.on_click(_guarded(_on_cancel))
schedule_btn.on_click(_guarded(_on_schedule))
free_btn.on_click(_guarded(_on_free))
floor_btn.on_click(_guarded(_on_floor))
ceiling_btn.on_click(_guarded(_on_ceiling))
brute_btn.on_click(_guarded(_on_brute))
range_btn.on_click(_guarded(_on_range))
load_btn.on_click(_guarded(_on_load))
reset_btn.on_click(_guarded(_on_reset))
clear_log_btn.on_click(_on_clear_log)
load_order.observe(_on_order_change, names="value")

ui = widgets.VBox([
    widgets.HTML(_CSS + _HEADER, layout=widgets.Layout(width="100%")),
    inputs,
    _group("Bookings &middot; Parts 1 &amp; 3"),
    _row([book_btn, lookup_btn, cancel_btn, schedule_btn, free_btn]),
    _group("Ordered queries &middot; Part 2"),
    _row([floor_btn, ceiling_btn, brute_btn, range_btn]),
    _group("Sample data"),
    _row([load_order, load_btn, reset_btn, clear_log_btn]),
    log_view,
    tree_view,
    widgets.HTML('<div class="ohl-foot">Buttons always run. An unfinished method just says so instead of crashing.</div>',
                 layout=widgets.Layout(width="100%")),
], layout=widgets.Layout(width="100%", max_width="1000px"))
ui.add_class("ohl-app")
_refresh()
display(ui)

## Lab tasks

10 questions, **10% each = 100%**. Do them in order. Every question needs
a screenshot (📸) where noted, and every question has a
**Your Answer Here** line right below it, replace that line with your
actual answer (for screenshot-only questions, a short "done, see
screenshot" is enough). Make sure `BSTMap` is fully implemented before
Part 1, `OrderedBSTMap` before Part 2, and `can_book` /
`next_free_slot` before Part 3. Clicking **Clear Log** before each
question keeps your screenshots tidy.

### Part 1: The tree (Questions 1-5)

**1. Load the schedule.** Make sure **Load order** is set to **Booking
order**, then click **Load Sample Data**. 📸 Screenshot the "Tree
internals" panel. Which booking ended up at the root, and why that one?
How many levels does the tree have, and what's the fewest levels 12 nodes
could possibly fit in?

**Your Answer Here**

**2. Look up, hit and miss.** Set Time to `15:15` and click **Look Up**.
Then set it to `15:20` and click **Look Up** again. 📸 Screenshot both
results in the action log. Which nodes did each search walk through?
Henry Scott is in the room at 15:20 (his meeting runs 15:15-15:30), so
why does Look Up say there's no booking?

**Your Answer Here**

**3. The schedule comes out sorted.** Click **Show Schedule**.
📸 Screenshot it. The bookings were inserted in a jumbled order and
nothing ever calls `sort()`, yet `__iter__` lists them in time order.
Why? (In the HashMap lab, List Roster had no meaningful order. What's
different here?)

**Your Answer Here**

**4. Cancel a leaf, then a node with one child.** Cancel `14:10`, then
cancel `13:00`. 📸 Screenshot the tree after each. For each one, say
which case of `__delitem__` it was, and which links changed (`_left` /
`_right` and `_parent`).

**Your Answer Here**

**5. Cancel a node with two children.** Cancel `14:30`, the root.
📸 Screenshot the tree. Which booking is at the root now, and why exactly
that one? Follow the blue numbers: which nodes did the delete walk
through to find it? Why is that booking's old node guaranteed to have no
left child?

**Your Answer Here**

### Part 2: Ordered queries (Questions 6-8)

Click **Load Sample Data** again first (booking order), so everyone
starts from the same tree.

**6. Who's in the room?** A student knocks on the door at `15:05`. Set
Time to `15:05` and click **Floor ≤**, then **Ceiling ≥**. 📸 Screenshot
both log lines. Who is in the room right now, and who's next? How many
nodes did each query look at?

**Your Answer Here**

**7. Search vs. ceiling.** Set Time to `13:01`. Click **Next ≥ using only
in**, then **Ceiling ≥**. 📸 Screenshot both log lines. How many `in`
calls, and how many nodes in total, did the first approach need, versus
`ceiling_key`? Now imagine start times were stored to the millisecond
instead of the minute: how many `in` calls would the same question take,
and what if they were floats? Why doesn't `ceiling_key` care?

**Your Answer Here**

**8. Range query with pruning.** Set Time to `15:00` and Until to
`15:30`, then click **Range**. 📸 Screenshot the tree panel. Which nodes
did `range_keys` look at, and which whole subtrees did it skip? Explain
why skipping them is safe.

**Your Answer Here**

### Part 3: Booking rules (Question 9)

**9. Clashes and openings.** Type your own name into Name, then:
(a) try to book `13:40` for 10 min, it should be refused;
(b) book `14:00` for 10 min, it should be accepted even though it touches
Daniel Lee's end and Emma Wilson's start;
(c) set Time to `13:40` and Length to 15 min, click **Next Free Slot**,
then **Book**. 📸 Screenshot the log showing all of it. `can_book` only
ever looks at two bookings, the floor and the ceiling of the new start
time. Why can't any *other* booking clash with the new meeting?

**Your Answer Here**

### Wrap-up (Question 10)

**10. When the tree goes wrong.** Click **Load Sample Data** (booking
order), set Time to `16:40`, click **Ceiling ≥**, and note how many nodes
it looked at. Now switch **Load order** to **Chronological**, click
**Load Sample Data**, and click **Ceiling ≥** at `16:40` again.
📸 Screenshot the tree. How many levels does it have now, and how many
nodes did the same query look at? Explain why inserting the bookings in
time order (exactly what a receptionist copying a paper schedule would
do) produces this shape, what it does to the cost of every operation, and
what kind of tree would avoid it.

**Your Answer Here**

**Submit:** attach this notebook (or your screenshots, in order) with
every **Your Answer Here** filled in.